# LeetCode #196: Delete Duplicate Emails

https://leetcode.com/problems/delete-duplicate-emails/

## Comparison of Approaches
| Approach | Time | Space |
|---|---|---|
| Self-Join DELETE ★ | O(n²) | O(1) |
| Subquery with MIN | O(n²) | O(n) |

## Understanding the Methods
### Self-Join DELETE (Optimal SQL)
Join the Person table with itself on email, then delete rows where the id is greater than the other matching id. This keeps the row with the smallest id for each email.

### Subquery with MIN
Find the minimum id for each email group, then delete all rows whose id is not in that set. Same result but uses a subquery.

## Solutions

### C#

In [ ]:
// SQL Problem — SQL solution:
/*
DELETE p1
FROM Person p1, Person p2
WHERE p1.email = p2.email AND p1.id > p2.id;
*/

// Alternative SQL:
/*
DELETE FROM Person
WHERE id NOT IN (
    SELECT * FROM (
        SELECT MIN(id) FROM Person GROUP BY email
    ) AS tmp
);
*/

// C# equivalent using LINQ:
using System;
using System.Collections.Generic;
using System.Linq;

var persons = new List<(int id, string email)> {
    (1, "john@example.com"),
    (2, "bob@example.com"),
    (3, "john@example.com")
};

var minIds = persons.GroupBy(p => p.email)
    .Select(g => g.Min(p => p.id))
    .ToHashSet();
persons.RemoveAll(p => !minIds.Contains(p.id));

foreach (var p in persons)
    Console.WriteLine($"id={p.id}, email={p.email}");

### Python

In [ ]:
import pandas as pd

def delete_duplicate_emails(person: pd.DataFrame) -> None:
    """
    SQL equivalent:
    DELETE p1 FROM Person p1, Person p2
    WHERE p1.email = p2.email AND p1.id > p2.id;
    """
    # Keep the row with the smallest id for each email
    min_ids = person.groupby('email')['id'].transform('min')
    drop_idx = person[person['id'] != min_ids].index
    person.drop(drop_idx, inplace=True)

### Go

In [ ]:
package main

import "fmt"

type Person struct {
    ID    int
    Email string
}

func deleteDuplicateEmails(persons []Person) []Person {
    minID := make(map[string]int)
    for _, p := range persons {
        if id, ok := minID[p.Email]; !ok || p.ID < id {
            minID[p.Email] = p.ID
        }
    }
    var result []Person
    for _, p := range persons {
        if p.ID == minID[p.Email] {
            result = append(result, p)
        }
    }
    return result
}

func main() {
    persons := []Person{{1, "john@example.com"}, {2, "bob@example.com"}, {3, "john@example.com"}}
    result := deleteDuplicateEmails(persons)
    for _, p := range result {
        fmt.Printf("id=%d, email=%s\n", p.ID, p.Email)
    }
}

### Rust

In [ ]:
use std::collections::HashMap;

#[derive(Debug)]
struct Person { id: i32, email: String }

fn delete_duplicate_emails(persons: Vec<Person>) -> Vec<Person> {
    let mut min_id: HashMap<&str, i32> = HashMap::new();
    for p in &persons {
        let entry = min_id.entry(&p.email).or_insert(p.id);
        if p.id < *entry {
            *entry = p.id;
        }
    }
    persons.into_iter()
        .filter(|p| min_id[p.email.as_str()] == p.id)
        .collect()
}

fn main() {
    let persons = vec![
        Person { id: 1, email: "john@example.com".into() },
        Person { id: 2, email: "bob@example.com".into() },
        Person { id: 3, email: "john@example.com".into() },
    ];
    let result = delete_duplicate_emails(persons);
    for p in &result {
        println!("id={}, email={}", p.id, p.email);
    }
}

## Examples

**Common:** Persons `[(1,john), (2,bob), (3,john)]` → delete id=3, keep `[(1,john), (2,bob)]`.

**Slightly Complex:** Three duplicates of same email with ids 5, 2, 8 → keep id=2, delete ids 5 and 8.

**Edge Case (No Duplicates):** All emails unique → nothing deleted.

**Edge Case (All Same):** Persons `[(1,a), (2,a), (3,a)]` → keep only id=1.

**Single Row:** Only one person in table → nothing deleted.

*Infographic will be added in a future update.*